# Eigenmode simulation of single qubit with readout resonator
This notebook was used to generate the study where the chi expression get_opt_target_res_qub_chi_via_coupl_length in src\qdesignoptimizer\utils\optimization_targets.py was manually taken to the power x=[1.0, 0.8,0.6,0.4,0.35,0.3]

In [ ]:
%load_ext autoreload
%autoreload 2

## Rendering the design

In [ ]:
import names as n
import design as d
from qdesignoptimizer.utils.chip_generation import create_chip_base

In [ ]:
design, gui = create_chip_base(
    n.CHIP_NAME,
    d.chip_type,
    open_gui=True,
    design_variables_file="design_variables.json",
)
d.render_qiskit_metal_design(design, gui)

## Creating the study and optimization targets

In [ ]:
import mini_studies as ms
import optimization_targets as ot

In [ ]:
MINI_STUDY_GROUP = n.NBR_1
MINI_STUDY = ms.get_mini_study_qb_res(group=MINI_STUDY_GROUP)
MINI_STUDY.build_fine_mesh = False
RENDER_QISKIT_METAL = lambda design: d.render_qiskit_metal_design(design, gui)

opt_targets = ot.get_opt_targets_2qubits_resonator_coupler(
    groups=[MINI_STUDY_GROUP],
    opt_target_qubit_freq=True,
    opt_target_qubit_anharm=True,
    opt_target_resonator_freq=True,
    opt_target_resonator_kappa=False,
    opt_target_resonator_qubit_chi=True,
    use_simple_resonator_qubit_chi_relation=False,
)

## Creating design analysis objects

In [ ]:
import parameter_targets as pt
import plot_settings as ps
from design import CoupledLineTee_mesh_names

from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

from qdesignoptimizer.design_analysis import DesignAnalysis, DesignAnalysisState
from qdesignoptimizer.design_analysis_types import MeshingMap
from qdesignoptimizer.utils.utils import get_save_path, close_ansys

close_ansys()

In [ ]:
design_analysis_state = DesignAnalysisState(
    design, RENDER_QISKIT_METAL, pt.PARAM_TARGETS
)

# map for finer meshing
meshing_map = [
    MeshingMap(component_class=CoupledLineTee, mesh_names=CoupledLineTee_mesh_names)
]

design_analysis = DesignAnalysis(
    design_analysis_state,
    mini_study=MINI_STUDY,
    opt_targets=opt_targets,
    save_path=get_save_path("out/", n.CHIP_NAME),
    update_design_variables=False,
    plot_settings=ps.PLOT_SETTINGS,
    meshing_map=meshing_map,
)

## Optimization step

In [ ]:
# number of runs of optimization and number of passes for simulation at each run
nbr_iterations = 10
group_passes = 12
delta_f = 0.001
for i in range(nbr_iterations):
    design_analysis.update_nbr_passes(group_passes)
    design_analysis.update_delta_f(delta_f)
    design_analysis.optimize_target({}, {})
    design_analysis.screenshot(gui=gui, run=i)